In [2]:
import os
import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font, PatternFill, Alignment
from datetime import datetime
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Function to find the file with "SDP - VM Analysis" in the title
def find_analysis_file(directory):
    for file in os.listdir(directory):
        if "SDP - VM Analysis" in file and file.endswith(".xlsx"):
            return os.path.join(directory, file)
    return None

# Function to extract client name from file name
def extract_client_name(file_name):
    return file_name.split("-")[0].strip()

# Helper function to get unique count for a column
def get_unique_count(sheet, column_name):
    df = pd.DataFrame(sheet.values)
    df.columns = df.iloc[0]  # Use first row as header
    df = df.drop(0)  # Drop the header row
    return df[column_name].nunique()

# Helper function to get unique combinations count for multiple columns
def get_unique_combinations_count(sheet, columns):
    df = pd.DataFrame(sheet.values)
    df.columns = df.iloc[0]  # Use first row as header
    df = df.drop(0)  # Drop the header row
    return df.drop_duplicates(subset=columns).shape[0]

def add_summary_sheet(wb, summary_data, valid_tax_id_count, hard_duplicates_count):
    summary_sheet = wb.create_sheet(title="Duplicate SUMMARY", index=0)
    summary_sheet.column_dimensions['A'].width = 25
    summary_sheet.column_dimensions['B'].width = 22
    summary_sheet.column_dimensions['C'].width = 20
    summary_sheet.column_dimensions['D'].width = 20

    # Define colors
    teal_fill = PatternFill(start_color="00FFFF", end_color="00FFFF", fill_type="solid")
    light_teal_fill = PatternFill(start_color="CCFFFF", end_color="CCFFFF", fill_type="solid")
    orange_fill = PatternFill(start_color="FFA500", end_color="FFA500", fill_type="solid")  # Orange highlight
    bold_font = Font(bold=True)
    right_align = Alignment(horizontal="right")  # Right alignment
    number_format = '#,##0'  # Number format with commas


    # Set headers with teal highlight and bold font
    headers = ["Type", "Total Uniques + Duplicates", "Uniques Only Count", "Duplicates Only Count"]
    for col_num, header in enumerate(headers, start=1):
        cell = summary_sheet.cell(row=1, column=col_num, value=header)
        cell.font = bold_font  # Make header bold
        cell.fill = teal_fill  # Apply teal fill to headers

    # Populate the summary data and apply light teal highlight to rows 2-6
    for row_num, (duplicate_type, total_count) in enumerate(summary_data.items(), start=2):
        summary_sheet.cell(row=row_num, column=1, value=duplicate_type)
        summary_sheet.cell(row=row_num, column=2, value=total_count)

        # Calculate unique and duplicate counts based on duplicate type from corresponding tabs
        if duplicate_type in wb.sheetnames:
            duplicate_sheet = wb[duplicate_type]
            if duplicate_type == "ID Duplicates":
                unique_count = get_unique_count(duplicate_sheet, "internal supplier id")
            elif duplicate_type == "Name Duplicates":
                unique_count = get_unique_count(duplicate_sheet, "company name")
            elif duplicate_type == "Address Duplicates":
                unique_count = get_unique_count(duplicate_sheet, "complete full address")
            elif duplicate_type == "Name.Address Duplicates":
                unique_count = get_unique_combinations_count(duplicate_sheet, ["company name", "complete full address"])
            elif duplicate_type == "Tax ID Duplicates":
                unique_count = get_unique_count(duplicate_sheet, "tax id")

            # Calculate duplicates only count
            duplicate_count = total_count - unique_count

            # Fill the new columns
            summary_sheet.cell(row=row_num, column=3, value=unique_count).number_format = number_format
            summary_sheet.cell(row=row_num, column=4, value=duplicate_count).number_format = number_format

        # Apply light teal highlight to rows 2-6
        if row_num <= 6:
            for col in range(1, 5):  # Columns A to D
                summary_sheet.cell(row=row_num, column=col).fill = light_teal_fill

        # Apply number format with commas to "Total Uniques + Duplicates"
        summary_sheet.cell(row=row_num, column=2).number_format = number_format

    # Handle "Hard Duplicate Rows Removed" entry with orange highlight
    hard_row = 7  # Row number for "Hard Duplicate Rows Removed"
    summary_sheet.cell(row=hard_row, column=1, value="Hard Duplicate Rows Removed")
    summary_sheet.cell(row=hard_row, column=2, value="n/a")  # Set Total Uniques + Duplicates to "n/a"
    summary_sheet.cell(row=hard_row, column=3, value="n/a")  # Set Uniques Only Count to "n/a"
    summary_sheet.cell(row=hard_row, column=4, value=hard_duplicates_count).number_format = number_format  # Set Duplicates Only Count to the number of rows

    # Apply right alignment to "n/a" values in columns B and C
    summary_sheet.cell(row=hard_row, column=2).alignment = right_align  # Align "n/a" to the right in column B
    summary_sheet.cell(row=hard_row, column=3).alignment = right_align  # Align "n/a" to the right in column C

    # Apply orange fill to A7:D7
    for col in range(1, 5):  # Columns A to D
        cell = summary_sheet.cell(row=hard_row, column=col)
        cell.fill = orange_fill
        cell.font = bold_font  # Make "Hard Duplicate Rows Removed" row bold


# Function to format duplicate sheets
def format_duplicate_sheet(sheet, highlight_columns):
    # Set column widths
    sheet.column_dimensions['A'].width = 20
    sheet.column_dimensions['B'].width = 30
    sheet.column_dimensions['C'].width = 50

    # Define styles
    teal_fill = PatternFill(start_color="00FFFF", end_color="00FFFF", fill_type="solid")
    light_teal_fill = PatternFill(start_color="CCFFFF", end_color="CCFFFF", fill_type="solid")
    bold_font = Font(bold=True)

    # Apply header formatting
    for cell in sheet[1]:
        cell.font = bold_font
        cell.fill = teal_fill

    # Apply column-specific formatting
    for row in sheet.iter_rows(min_row=2, min_col=1, max_col=len(sheet[1])):
        for col_num, cell in enumerate(row, start=1):
            if col_num in highlight_columns:
                cell.fill = light_teal_fill
    # Apply light teal fill to "YES" in "Possible valid Tax ID?"
    if sheet.cell(row=1, column=5).value == "Possible valid Tax ID?":
        for row in sheet.iter_rows(min_row=2, min_col=5, max_col=5, max_row=sheet.max_row):
            for cell in row:
                if cell.value == "YES":
                    cell.fill = light_teal_fill

def create_duplicate_tabs(df, wb):
    summary_data = {
        "ID Duplicates": 0,
        "Name Duplicates": 0,
        "Address Duplicates": 0,
        "Name.Address Duplicates": 0,
        "Tax ID Duplicates": 0
    }
    valid_tax_id_count = 0

    # Helper function to add duplicates to sheet
    def add_duplicates_to_sheet(sheet, duplicates_df, columns):
        for col_num, column_title in enumerate(columns, start=1):
            sheet.cell(row=1, column=col_num, value=column_title)
        for row_num, row_data in enumerate(duplicates_df.values, start=2):
            for col_num, cell_value in enumerate(row_data, start=1):
                sheet.cell(row=row_num, column=col_num, value=cell_value)

    # Check for duplicates in different columns and sort them
    duplicate_conditions = [
        ("ID Duplicates", ["internal supplier id", "company name", "complete full address"], "internal supplier id", [1]),
        ("Name Duplicates", ["internal supplier id", "company name", "complete full address"], "company name", [2]),
        ("Address Duplicates", ["internal supplier id", "company name", "complete full address"], "complete full address", [3]),
        ("Name.Address Duplicates", ["internal supplier id", "company name", "complete full address"], ["company name", "complete full address"], [2, 3]),
        ("Tax ID Duplicates", ["internal supplier id", "company name", "complete full address", "tax id"], "tax id", [4])
    ]

    for sheet_name, columns, duplicate_by, highlight_columns in duplicate_conditions:
        sheet = wb.create_sheet(title=sheet_name)
        if sheet_name == "Tax ID Duplicates" and "tax id" not in df.columns:
            for col_num, column_title in enumerate(columns, start=1):
                sheet.cell(row=1, column=col_num, value=column_title)
            sheet.cell(row=2, column=1, value="NO Tax IDs")
        else:
            # Convert relevant columns to lowercase and strip spaces (case-insensitive matching)
            df_normalized = df.copy()
            if isinstance(duplicate_by, list):
                for col in duplicate_by:
                    df_normalized[col] = df_normalized[col].astype(str).str.lower().str.strip()
            else:
                df_normalized[duplicate_by] = df_normalized[duplicate_by].astype(str).str.lower().str.strip()

            # Identify duplicates
            duplicates_df = df_normalized[df_normalized.duplicated(duplicate_by, keep=False)]
            duplicates_df = duplicates_df[columns]

            # Sort based on specific column(s)
            if isinstance(duplicate_by, list):
                duplicates_df = duplicates_df.sort_values(by=duplicate_by)
            else:
                duplicates_df = duplicates_df.sort_values(by=[duplicate_by])

            # Add sorted duplicates to the sheet
            add_duplicates_to_sheet(sheet, duplicates_df, columns)
            
            # Update summary data
            summary_data[sheet_name] = len(duplicates_df)

            # Format the duplicate sheet
            format_duplicate_sheet(sheet, highlight_columns)

    return summary_data, valid_tax_id_count

# Function to add "Hard Duplicate Rows Removed" to summary
def add_hard_duplicates_summary(wb, hard_duplicates_count):
    summary_sheet = wb['Duplicate SUMMARY']
    summary_sheet.cell(row=7, column=1, value="Hard Duplicate Rows Removed")
    summary_sheet.cell(row=7, column=2, value=hard_duplicates_count)
    # Highlight the new row
    summary_sheet.cell(row=7, column=1).fill = PatternFill(start_color="00FFFF", end_color="00FFFF", fill_type="solid")
    summary_sheet.cell(row=7, column=2).fill = PatternFill(start_color="00FFFF", end_color="00FFFF", fill_type="solid")

# Function to count and remove hard duplicates
def count_and_remove_hard_duplicates(df):
    # Define the columns to check for hard duplicates
    hard_duplicate_columns = ['internal supplier id', 'company name', 'complete full address']
    
    # Create a cleaned DataFrame for deduplication (case-insensitive)
    df_cleaned = df.copy()
    for col in hard_duplicate_columns:
        df_cleaned[col] = df_cleaned[col].astype(str).fillna('').apply(lambda x: ''.join(e for e in x if e.isprintable())).str.strip().str.lower()

    # Identify all hard duplicates (ignore first occurrences)
    hard_duplicates = df_cleaned[df_cleaned.duplicated(subset=hard_duplicate_columns, keep='first')]

    # Count total rows removed
    hard_duplicates_count = len(hard_duplicates)  # Count remaining duplicate rows

    # Filter the original DataFrame to keep only the hard duplicates
    hard_duplicates_df = df[df.index.isin(hard_duplicates.index)][hard_duplicate_columns]

    # Remove hard duplicates from the original DataFrame
    df_cleaned = df.drop(hard_duplicates.index)

    logging.info(f"Hard duplicates identified and removed: {hard_duplicates_count} rows")

    # Return the final DataFrame without duplicates and the duplicates DataFrame
    return df_cleaned, hard_duplicates_count, hard_duplicates_df

# Function to create a new Excel file with renamed columns from the "Vendor Master" tab
def create_excel_from_vendor_master(df, client_name, input_df):
    # Select required columns
    selected_columns = ['internal supplier id', 'company name', 'complete full address']
    df_selected = df[selected_columns]

    # Remove hard duplicates based on specific columns
    df_selected = df_selected.drop_duplicates(subset=selected_columns)

    # Rename 'complete full address' to 'complete address'
    df_selected.rename(columns={'complete full address': 'complete address'}, inplace=True)

    # Initialize the "web_domain" column with blanks
    df_selected['web_domain'] = ''

    # Step 1: Check if "website domain" exists, and merge if available (take the first instance only)
    if 'website domain' in input_df.columns:
        # Keep only the first occurrence for each 'internal supplier id'
        web_domain_df = input_df[['internal supplier id', 'website domain']].drop_duplicates(subset='internal supplier id')
        df_selected = df_selected.merge(
            web_domain_df,
            on='internal supplier id', how='left'
        )
        # Populate "web_domain" with data from "website domain" where available
        df_selected['web_domain'] = df_selected['website domain'].fillna('')

    # Step 2: If "contact email" is available, use it as a fallback (first occurrence only)
    if 'contact email' in input_df.columns:
        contact_email_df = input_df[['internal supplier id', 'contact email']].drop_duplicates(subset='internal supplier id')
        df_selected = df_selected.merge(
            contact_email_df,
            on='internal supplier id', how='left'
        )
        # Fill in "web_domain" with "contact email" where "web_domain" is still blank
        df_selected['web_domain'] = df_selected['web_domain'].replace('', pd.NA).fillna(df_selected['contact email']).fillna('')

##01.06.2025 update
    # Initialize the workbook
    wb = Workbook()
    ws = wb.active
    ws.title = "Vendor Master"

    # Add data to the workbook
    for row in dataframe_to_rows(df_selected, index=False, header=True):
        ws.append(row)

    # Pass workbook to process Step 1.1 and Step 1.2
    process_sdp_input_file(df_selected, client_name, wb)

    # Step 4: Filter out forbidden domains if "forbidden.xlsx" is available
    try:
        forbidden_domains_df = pd.read_excel("forbidden.xlsx", usecols=["Domain"])
        forbidden_domains_set = set(forbidden_domains_df['Domain'].str.lower().dropna())
        df_selected['web_domain'] = df_selected['web_domain'].apply(
            lambda x: None if x in forbidden_domains_set else x
        )
    except FileNotFoundError:
        logging.warning("The 'forbidden.xlsx' file was not found. Skipping forbidden domain filtering.")

    # Step 5: Drop unnecessary columns added during merging, if they exist
    df_selected.drop(columns=['website domain', 'contact email'], errors='ignore', inplace=True)

    # Create a new workbook and add the dataframe to a sheet
    wb = Workbook()
    ws = wb.active
    ws.title = "Vendor Master"

    # Append the dataframe rows to the worksheet
    for row in dataframe_to_rows(df_selected, index=False, header=True):
        ws.append(row)

# Step 3: Clean the "web_domain" column
def clean_web_domain(value):
    if pd.isna(value) or value == '':
        return None
    value = str(value).lower()  # Ensure value is a string

    # Step 1: Remove "http://www.", "https://www.", "http://", and "https://"
    value = value.replace("http://www.", "").replace("https://www.", "").replace("https://", "").replace("http://", "")

    # Step 2: Remove "@" characters
    if "@" in value:
        value = value.split("@")[-1]

    # Step 3: Remove everything after and including a "/" character
    if "/" in value:
        value = value.split("/")[0]

    # Step 4: Remove invalid or unwanted domains
    if "mail" in value or ";" in value or ".edu" in value or "." not in value:
        return None

    return value.strip()
        
##01.06.2025 update
def process_sdp_input_file(df_selected, client_name, wb):
    """
    Process the "SDP Input" file to handle renaming and creating Step 1.1 and Step 1.2 files.
    Always remove 'web_domain', 'website domain', and 'contact email' columns from Step 1.1 if Step 1.2 is created.
    """
    # Generate the initial file name with "Step 1"
    today = datetime.today().strftime('%Y-%m-%d')
    output_excel_file = f"{client_name} - SDP Input - {today}.xlsx"
    step_1_1_file = output_excel_file.replace("Step 1", "Step 1.1")
    step_1_2_file = output_excel_file.replace(f"{today}", f"{today} - Client Domains").replace("Step 1", "Step 1.2")

    # Define header styles
    teal_fill = PatternFill(start_color="00FFFF", end_color="00FFFF", fill_type="solid")
    bold_font = Font(bold=True)
    
    # Save Step 1.1 file initially
    wb.save(step_1_1_file)
    logging.info(f"Step 1.1 file saved: {step_1_1_file}")

    # Adjust column widths for Step 1.1
    ws_step_1_1 = wb.active
    ws_step_1_1.column_dimensions['A'].width = 16
    ws_step_1_1.column_dimensions['B'].width = 35
    ws_step_1_1.column_dimensions['C'].width = 80
    for cell in ws_step_1_1[1]:  # Style header row
        cell.fill = teal_fill
        cell.font = bold_font
    wb.save(step_1_1_file)
    logging.info(f"Column widths adjusted for Step 1.1: {step_1_1_file}")

    # Check if 'web_domain' column contains any non-empty values
    if not df_selected['web_domain'].isna().all():
        # Clean the 'web_domain' column format again before saving Step 1.2
        df_selected['web_domain'] = df_selected['web_domain'].apply(clean_web_domain)

        # Create and save Step 1.2 file
        wb_step_1_2 = Workbook()
        ws_step_1_2 = wb_step_1_2.active
        ws_step_1_2.title = "Vendor Master"
        for row in dataframe_to_rows(df_selected, index=False, header=True):
            ws_step_1_2.append(row)
        wb_step_1_2.save(step_1_2_file)
        logging.info(f"Step 1.2 file created with cleaned 'web_domain': {step_1_2_file}")

        # Adjust column widths and style headers for Step 1.2
        ws_step_1_2.column_dimensions['A'].width = 16
        ws_step_1_2.column_dimensions['B'].width = 35
        ws_step_1_2.column_dimensions['C'].width = 80
        ws_step_1_2.column_dimensions['D'].width = 30
        for cell in ws_step_1_2[1]:  # Style header row
            cell.fill = teal_fill
            cell.font = bold_font
        wb_step_1_2.save(step_1_2_file)
        logging.info(f"Step 1.2 file created with adjusted column widths and header styles: {step_1_2_file}")
        
        # Remove 'web_domain', 'website domain', and 'contact email' columns from Step 1.1 DataFrame
        df_step_1_1 = df_selected.drop(columns=['web_domain', 'website domain', 'contact email'], errors='ignore')

        # Overwrite Step 1.1 file without the specified columns
        wb_step_1_1 = Workbook()
        ws_step_1_1 = wb_step_1_1.active
        ws_step_1_1.title = "Vendor Master"
        for row in dataframe_to_rows(df_step_1_1, index=False, header=True):
            ws_step_1_1.append(row)
        ws_step_1_1.column_dimensions['A'].width = 16    
        ws_step_1_1.column_dimensions['B'].width = 35
        ws_step_1_1.column_dimensions['C'].width = 80
        for cell in ws_step_1_1[1]:  # Style header row
            cell.fill = teal_fill
            cell.font = bold_font
        wb_step_1_1.save(step_1_1_file)
        logging.info(f"'web_domain', 'website domain', and 'contact email' columns removed, Step 1.1 file updated: {step_1_1_file}")

# Function to format the "Hard Duplicates - Removed" sheet
def format_hard_duplicates_sheet(sheet):
    # Define styles
    teal_fill = PatternFill(start_color="00FFFF", end_color="00FFFF", fill_type="solid")
    light_teal_fill = PatternFill(start_color="CCFFFF", end_color="CCFFFF", fill_type="solid")
    bold_font = Font(bold=True)

    # Apply header formatting for columns D, E, and F (columns 4, 5, 6 in Excel)
    for col in [1, 2, 3]:  # Columns 1, 2, 3 correspond to A, B, C
        cell = sheet.cell(row=1, column=col)
        cell.font = bold_font
        cell.fill = teal_fill
        sheet.column_dimensions[chr(64 + col)].width = 35  # Set column width to 35

    # Apply light teal fill to the inputs below headers A, B, and C
    for row in sheet.iter_rows(min_row=2, min_col=1, max_col=3, max_row=sheet.max_row):
        for cell in row:
            cell.fill = light_teal_fill

# Main script
def main():
    directory = os.getcwd()  # Get the current working directory
    analysis_file_path = find_analysis_file(directory)
    
    if not analysis_file_path:
        logging.error("No file with 'SDP - VM Analysis' in the title found.")
        return

    logging.info(f"Found analysis file: {analysis_file_path}")

    # Load the "Vendor Master" sheet from the "SDP - VM Analysis" file
    analysis_wb = load_workbook(analysis_file_path, data_only=True)
    if 'Vendor Master' not in analysis_wb.sheetnames:
        logging.error("The 'SDP - VM Analysis' file does not contain a 'Vendor Master' sheet.")
        return
    
    vendor_master_ws = analysis_wb['Vendor Master']
    data = vendor_master_ws.values
    columns = next(data)
    df = pd.DataFrame(data, columns=[col.lower() for col in columns])

    logging.info(f"Columns in the Vendor Master sheet: {df.columns.tolist()}")

    # Extract client name from the file name
    file_name = os.path.basename(analysis_file_path)
    client_name = extract_client_name(file_name)
    logging.info(f"Client name extracted: {client_name}")

    # Step 1: Count and remove hard duplicates
    df_cleaned, hard_duplicates_count, hard_duplicates_df = count_and_remove_hard_duplicates(df)

    # Step 2: Create a new workbook
    wb = Workbook()
    wb.remove(wb.active)  # Remove the default sheet

    # Step 3: Add hard duplicates to a new sheet
    hard_duplicates_sheet = wb.create_sheet(title="Hard Duplicates - Removed")
    for row in dataframe_to_rows(hard_duplicates_df, index=False, header=True):
        hard_duplicates_sheet.append(row)
    
    # Format the "Hard Duplicates - Removed" sheet
    format_hard_duplicates_sheet(hard_duplicates_sheet)

    # Step 4: Use cleaned DataFrame (with hard duplicates removed) for remaining duplicate checks
    summary_data, valid_tax_id_count = create_duplicate_tabs(df_cleaned, wb)

    # Step 5: Add summary sheet
    add_summary_sheet(wb, summary_data, valid_tax_id_count, hard_duplicates_count)

    # Step 6: Save the new workbook with the specified name
    today = datetime.today().strftime('%Y-%m-%d')
    output_file = f"Step 2_{client_name} - Deduplication Report - {today}.xlsx"
    wb.save(output_file)
    logging.info(f"New Excel file created: {output_file}")

    # Step 7: Create Excel file from "Vendor Master" tab
    create_excel_from_vendor_master(df_cleaned, client_name, df)  # Pass df as input_df

# Run the main function
if __name__ == "__main__":
    main()

2025-01-16 10:40:23,696 - INFO - Found analysis file: /Users/louis.standridge/Desktop/Peloton - SDP - 01.2025/TEST FOLDER/Step 1_peloton - SDP - VM Analysis - 2025-01-02.xlsx
2025-01-16 10:40:23,878 - INFO - Columns in the Vendor Master sheet: ['code', 'spendcountry', 'sanctioned country', 'internal supplier id', 'company name', 'complete full address', 'contact email', 'website domain', 'tax id', 'po box', 'unedited company name', 'internal supplier id or vendor number', 'address (street name and number)', 'address (city)', 'address (state/province)', 'address (country)', 'address (zip code / postal code)', 'diversity status', 'preferred supplier status', 'naics code', 'duns number', 'lei number']
2025-01-16 10:40:23,878 - INFO - Client name extracted: Step 1_peloton
2025-01-16 10:40:23,901 - INFO - Hard duplicates identified and removed: 0 rows
2025-01-16 10:40:23,942 - INFO - New Excel file created: Step 2_Step 1_peloton - Deduplication Report - 2025-01-16.xlsx
2025-01-16 10:40:23,9